In [ ]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import hydroeval as he
import folium
from branca.colormap import linear
from branca.colormap import LinearColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime

In [ ]:
pathData = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/SamplingInt_Timeseries_NADPsites'
os.chdir(pathData)
inorganics = pd.read_csv('PREC_SO4_NO3_NH4_Timeseries.SamplingInt_Summed_pairedNADPsites.csv')
inorganics = inorganics[['siteId', 'dateOn', 'dateOff', 'SamplingInt', 'lat', 'lon', 'longitude', 'PREC_tot_mm','ppt_mm',
       'Mod_SO4_mgm2', 'Mod_NO3_mgm2', 'Mod_NH4_mgm2','SO4_mgm2', 'NO3_mgm2', 'NH4_mgm2']]
inorganics.head()

In [ ]:
#Conduct an analysis on precip accuracy
#Step 1: Remove rows where there is no observed rain but the model is predicting rain
nadp_df = nadp_df.loc[nadp_df.ppt_mm != 0,]

#calculate the percent error between modelled and observed (|modelled - observed|/modelled)*100%
nadp_df['Precip_Perc_Error'] = ((abs(nadp_df['PREC_tot_mm'] - nadp_df['ppt_mm']))/(nadp_df['PREC_tot_mm']))*100

#Create a categorical variable that is reflective of precip accuracy
conditions = [
    (nadp_df['Precip_Perc_Error'] >= 0) & (nadp_df['Precip_Perc_Error'] < 11),
    (nadp_df['Precip_Perc_Error'] >= 11) & (nadp_df['Precip_Perc_Error'] < 21)
]

labels = ['Excellent', 'Good']

# Create a new variable based on conditions
nadp_df['Precip_Quality'] = np.select(conditions, labels, default='Poor')
nadp_df.head()

In [ ]:
#Descriptive statistics on precip quality
precip_df = nadp_df.groupby(['siteId', 'lat', 'longitude'])[['Precip_Quality']].value_counts().unstack()
precip_df['Total'] = precip_df['Excellent'] + precip_df['Good'] + precip_df['Poor']
precip_df['Excellent'] = ((precip_df['Excellent'] / precip_df['Total']) * 100)
precip_df['Good'] = ((precip_df['Good'] / precip_df['Total']) * 100)
precip_df['Poor'] = ((precip_df['Poor'] / precip_df['Total']) * 100)
precip_df['Acceptable_20'] = precip_df['Excellent'] + precip_df['Good']
precip_df['Acceptable_10'] = precip_df['Excellent']

precip_df = precip_df.reset_index()
precip_df.head()

In [ ]:
# Create a figure and axis with a specific projection
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

# Add coastlines and other features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAND, edgecolor='black')
ax.add_feature(cfeature.OCEAN, edgecolor='black')

# Plot points with color based on 'Acceptable' values
sc = ax.scatter(precip_df['longitude'], precip_df['lat'], c=precip_df['Acceptable_10'], cmap='YlOrRd', s=30, alpha=0.7)

# Add colorbar
plt.colorbar(sc, ax=ax, label='Acceptable_10')

# Show the plot
plt.show()